<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_(B)_%5BDUTCH%5D_Hate_Speech_Classification_part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [2]:
import pandas as pd
import numpy as np
import torch

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
df_Dh = pd.read_csv('/content/StereoQueerEval_NL_training.tsv', sep='\t')
df_Dh.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_NL_0001,Non-binaire mensen kunnen binnenkort vragen om...,Binnenkort kunnen non-binaire mensen vragen om...,een schande dat ons geld hier naartoe gaat,no,yes_implicit,group_nb
1,training_NL_0002,Non-binaire mensen kunnen binnenkort vragen om...,Binnenkort kunnen non-binaire mensen vragen om...,"Ach shit,...et is idd 1 april natuurlijk😂....",no,yes_implicit,group_nb
2,training_NL_0003,Acceptatie lhbti+ op dieptepunt! ’Social media...,Ooit was Nederland een tolerant land en de hom...,"Nee, dit accepteren we niet.",no,yes_implicit,group_lgbtqia+
3,training_NL_0004,Pride Amsterdam staat onder spanning: ‘Ermee s...,Amsterdam - ooit de gay-capital maar anno nu a...,Op pride en lhbti gemeenschap mogen het zo gek...,yes,yes_explicit,group_lgbtqia+
4,training_NL_0005,Gehandicapte transgender rapper Arjen wil Nede...,Wij vonden een opmerkelijke verhaal in onze ma...,Een travestiet is niet hetzelfde als een trans...,no,no,none


In [5]:
df = df_Dh

In [6]:
df.shape

(2238, 7)

In [7]:
df['hate_speech'].value_counts()

,count
hate_speech,
no,1127
yes_explicit,624
yes_implicit,487


In [8]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [9]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

#Experiment 3.A
oly comment as input

In [ ]:
X = (df.drop(['StereoQueerEval_id', 'yt_title', 'yt_description', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2})

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=45)
X_val, X_test, y_val, y_test = train_test_split(X, y, test_size = 0.2, random_state=45)

In [ ]:
def build_inputs(d):
    comments = d['yt_comment'].to_list()
    return comments

def encode(d, tokenizer, max_length=256):
    comments = build_inputs(d)
    return tokenizer(comments, truncation=True, max_length=max_length)

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###Distilbert-Multilingual

In [ ]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.118003
20,1.097602
30,1.084699
40,1.067296
50,0.992476
60,1.029041
70,1.054216
80,1.011248
90,1.051511
100,0.956045


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.7044669936810221, metrics={'train_runtime': 243.5316, 'train_samples_per_second': 36.751, 'train_steps_per_second': 4.599, 'total_flos': 253325649718224.0, 'train_loss': 0.7044669936810221, 'epoch': 5.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.676     0.724     0.699       210
yes_explicit      0.515     0.519     0.517       129
yes_implicit      0.452     0.385     0.416       109

    accuracy                          0.583       448
   macro avg      0.548     0.543     0.544       448
weighted avg      0.575     0.583     0.578       448

[[152  28  30]
 [ 41  67  21]
 [ 32  35  42]]


###twitter-roberta-base-hate-latest

In [ ]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.062513
20,1.021593
30,0.980030
40,1.067645
50,1.040795
60,0.982604
70,1.077969
80,1.028801
90,0.981794
100,0.986486


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.8370001230921064, metrics={'train_runtime': 395.625, 'train_samples_per_second': 22.622, 'train_steps_per_second': 1.415, 'total_flos': 827468059658436.0, 'train_loss': 0.8370001230921064, 'epoch': 5.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.636     0.724     0.677       210
    explicit      0.444     0.488     0.465       129
    implicit      0.478     0.294     0.364       109

    accuracy                          0.551       448
   macro avg      0.519     0.502     0.502       448
weighted avg      0.542     0.551     0.540       448

[[152  40  18]
 [ 49  63  17]
 [ 38  39  32]]


#Experiment 3.B
comment and title as input

In [10]:
X = (df.drop(['StereoQueerEval_id', 'yt_description', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2})

In [12]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [13]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [14]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###twitter-roberta-base-hate-latest

In [15]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [16]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [18]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [19]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.131023
20,1.064174
30,0.982087
40,1.072727
50,1.020840
60,0.973510
70,1.051457
80,1.060570
90,0.982907
100,0.975267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.8505630101476397, metrics={'train_runtime': 363.1993, 'train_samples_per_second': 24.642, 'train_steps_per_second': 1.542, 'total_flos': 906465807757692.0, 'train_loss': 0.8505630101476397, 'epoch': 5.0})

In [20]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.726     0.783     0.753       115
    explicit      0.464     0.500     0.481        52
    implicit      0.409     0.316     0.356        57

    accuracy                          0.598       224
   macro avg      0.533     0.533     0.530       224
weighted avg      0.585     0.598     0.589       224

[[90 11 14]
 [14 26 12]
 [20 19 18]]


###distilbert-base-multilingual-cased

In [21]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [22]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [24]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.110603
20,1.088854
30,1.074531
40,1.061151
50,0.951767
60,1.030944
70,1.093327
80,1.015803
90,1.064115
100,0.978041


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.773248211826597, metrics={'train_runtime': 167.4104, 'train_samples_per_second': 53.461, 'train_steps_per_second': 6.69, 'total_flos': 293222333486628.0, 'train_loss': 0.773248211826597, 'epoch': 5.0})

In [26]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.702     0.757     0.728       115
yes_explicit      0.448     0.500     0.473        52
yes_implicit      0.476     0.351     0.404        57

    accuracy                          0.594       224
   macro avg      0.542     0.536     0.535       224
weighted avg      0.585     0.594     0.586       224

[[87 17 11]
 [15 26 11]
 [22 15 20]]


# Experiment 3.C
comment + title + description as input

In [7]:
X = (df.drop(['StereoQueerEval_id', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({"no": 0, "yes_explicit": 1, "yes_implicit": 2})

In [8]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [9]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    desc  = d['yt_description'].fillna("").str[:300]        # trim long descriptions
    ctx = (title + " . " + desc)
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [10]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###distilbert-base-multilingual-cased

In [11]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [12]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [14]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.096981
20,1.097799
30,1.065919
40,1.058041
50,0.966110
60,1.037799
70,1.084941
80,1.027798
90,1.059179
100,0.977672


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.8789806740624564, metrics={'train_runtime': 271.3353, 'train_samples_per_second': 32.985, 'train_steps_per_second': 4.128, 'total_flos': 463369392321876.0, 'train_loss': 0.8789806740624564, 'epoch': 5.0})

In [16]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.685     0.774     0.727       115
yes_explicit      0.463     0.481     0.472        52
yes_implicit      0.450     0.316     0.371        57

    accuracy                          0.589       224
   macro avg      0.533     0.523     0.523       224
weighted avg      0.573     0.589     0.577       224

[[89 13 13]
 [18 25  9]
 [23 16 18]]


###twitter-roberta-base-hate-latest

In [17]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [19]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [20]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [21]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
10,1.072451
20,1.025581
30,0.978394
40,1.051093
50,0.986579
60,0.995842
70,1.095863
80,1.049381
90,0.976888
100,0.946394


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.8738465888159616, metrics={'train_runtime': 415.9535, 'train_samples_per_second': 21.517, 'train_steps_per_second': 1.346, 'total_flos': 1155346434045468.0, 'train_loss': 0.8738465888159616, 'epoch': 5.0})

In [22]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.683     0.843     0.755       115
    explicit      0.408     0.385     0.396        52
    implicit      0.485     0.281     0.356        57

    accuracy                          0.594       224
   macro avg      0.525     0.503     0.502       224
weighted avg      0.569     0.594     0.570       224

[[97 12  6]
 [21 20 11]
 [24 17 16]]
